In [1]:
# importing libraries

import pandas as pd
import numpy as np
import os
import sklearn
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [2]:
# importing Titanic dataset

df_titanic = pd.read_csv(r"C:\\Users\\User\\2_ANN_model_learning_scripts\\0_ANN_tabular_data\\Titanic_dataset.csv")

# printing basic informations such as number of observations and number of features

print(" Number of observations:", df_titanic.shape[0],'\n',"Number of feature:", df_titanic.shape[1],'\n')

# printing top 5 observations

df_titanic.head(5)

 Number of observations: 891 
 Number of feature: 12 



,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
# creating a copy of the actual data

df_titanic_relvent = df_titanic.copy()

# dropping un-necessory features

df_titanic_relvent.drop(columns = {'PassengerId', 'Name', 'Ticket', 'Cabin'}, inplace = True)

# changing the dependent variable name

df_titanic_relvent = df_titanic_relvent.rename(columns = {'Survived':'Target'})

# priniting top 5 observations

df_titanic_relvent.head()

,Target,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [4]:
# categorical treatment - OHE

categorical_cols = ['Pclass', 'Sex', 'Embarked']

for col in categorical_cols:
    df_titanic_relvent = pd.get_dummies(df_titanic_relvent, columns=[col], dtype=int)

# MinMaxScaler on continous variable

continous_cols = ['Age', 'SibSp', 'Fare']

for col in continous_cols:
    df_titanic_relvent[col] = MinMaxScaler().fit_transform(df_titanic_relvent[[col]])

# replacing null values with 0

cols = list(df_titanic_relvent.columns)
cols.remove('Target')

for col in cols:
    df_titanic_relvent[col] = df_titanic_relvent[col].fillna(0)
    
# train test split

X = df_titanic_relvent.drop(columns = 'Target')
y = df_titanic_relvent['Target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state= 10, shuffle= True)

### perceptron model

In [5]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

In [6]:
# perceptron uses a single layer, hence we uses a single dense layer only

MLP_model = Sequential([Dense(units = 16, activation= 'relu', name = 'layer1'),
                               Dense(units = 8 , activation= 'relu', name = 'layer2'),
                               Dense(units = 4 , activation= 'relu', name = 'layer3'),
                               Dense(units = 2 , activation= 'relu', name = 'layer4'),
                               Dense(units = 1 , activation= 'sigmoid', name = 'layer5')
                              ])

In [7]:
# model compiling steps: 
# Optimizer - "sgd" (stochastic gradient descent) 
# Loss function - binary_crossentropy 
# metrics  = accuracy

MLP_model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

In [8]:
# Applying the early stopping concept

early_stopping = EarlyStopping(monitor='val_loss',
    patience=50,
    min_delta=0.1,
    restore_best_weights=True,
    verbose=1
)

In [9]:
# fitting the model using train data

MLP_model.fit(X_train, y_train, 
              validation_data= (X_test, y_test), 
              epochs=500, 
              batch_size = 128,
             callbacks=[early_stopping])

Epoch 1/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 45ms/step - accuracy: 0.3997 - loss: 0.6971 - val_accuracy: 0.4469 - val_loss: 0.6944
Epoch 2/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.4896 - loss: 0.6917 - val_accuracy: 0.6369 - val_loss: 0.6901
Epoch 3/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6538 - loss: 0.6882 - val_accuracy: 0.6592 - val_loss: 0.6864
Epoch 4/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.6906 - loss: 0.6831 - val_accuracy: 0.6983 - val_loss: 0.6798
Epoch 5/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6656 - loss: 0.6745 - val_accuracy: 0.7039 - val_loss: 0.6725
Epoch 6/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6686 - loss: 0.6704 - val_accuracy: 0.7207 - val_loss: 0.6654
Epoch 7/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7195 - loss: 0.6589 - val_accuracy: 0.7542 - val_loss: 0.6587
Epoch 8/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7548 - loss: 0.6539 - val_accuracy: 0.7709 - val_loss:

Epoch 52/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8133 - loss: 0.5256 - val_accuracy: 0.8101 - val_loss: 0.5463
Epoch 53/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8075 - loss: 0.5442 - val_accuracy: 0.8101 - val_loss: 0.5435
Epoch 54/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8027 - loss: 0.5366 - val_accuracy: 0.8101 - val_loss: 0.5428
Epoch 55/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8149 - loss: 0.5259 - val_accuracy: 0.8101 - val_loss: 0.5420
Epoch 56/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8293 - loss: 0.5162 - val_accuracy: 0.8101 - val_loss: 0.5429
Epoch 57/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8277 - loss: 0.5114 - val_accuracy: 0.8101 - val_loss: 0.5407
Epoch 58/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8120 - loss: 0.5234 - val_accuracy: 0.8101 - val_loss: 0.5387
Epoch 59/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.8124 - loss: 0.5124 - val_accuracy: 0.8101 - v

In [10]:
# prediction using X_test

y_pred_test = MLP_model.predict(X_test)

# converting observations 1 if prediction is more than 0.5

y_pred_test = (y_pred_test > 0.5).astype(int)

# calculating the accuracy on test data

accuracy_score(y_test, y_pred_test)

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step


0.8212290502793296

In [11]:
# to get the model summary

MLP_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ layer1 (Dense)                       │ (None, 16)                  │             208 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ layer2 (Dense)                       │ (None, 8)                   │             136 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ layer3 (Dense)                       │ (None, 4)                   │              36 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ layer4 (Dense)                       │ (None, 2)                   │              10 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ layer5 (Dense)                       │ (None, 1)                   │               3 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,181 (4.62 KB)

 Trainable params: 393 (1.54 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 788 (3.08 KB)